In [119]:
import kagglehub
import tensorflow as tf
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split
import keras

In [120]:
base_dir='/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset'

train_dir = os.path.join(base_dir, 'Training')
validation_dir = os.path.join(base_dir, 'Testing')

train_glioma_dir = os.path.join(train_dir, 'glioma')
train_meningioma_dir = os.path.join(train_dir, 'meningioma')
train_pituitary_dir = os.path.join(train_dir, 'pituitary')
train_notumor_dir = os.path.join(train_dir, 'notumor')

validation_glioma_dir = os.path.join(validation_dir, 'glioma')
validation_meningioma_dir = os.path.join(validation_dir, 'meningioma')
validation_pituitary_dir = os.path.join(validation_dir, 'pituitary')
validation_notumor_dir = os.path.join(validation_dir, 'notumor')

In [121]:
image_size = (128,128)
Batch_size = 32

In [122]:
def load_images_and_labels(directory , label):
    images = []
    labels = []

    image_files = [f for f in os.listdir(directory) if f.lower().endswith(('.png' , '.jpg' , '.jpeg'))]
    for filename in image_files:
        image_path = os.path.join(directory , filename)

        image = cv2.imread(image_path)
        image = cv2.resize(image, image_size)

        images.append(image)
        labels.append(label)
    return images , labels

In [123]:
x_train_glioma , y_train_glioma = load_images_and_labels( train_glioma_dir , 0 )
x_train_meningioma , y_train_meningioma = load_images_and_labels( train_meningioma_dir , 1 )
x_train_pituitary , y_train_pituitary = load_images_and_labels( train_pituitary_dir , 2 )
x_train_notumor , y_train_notumor = load_images_and_labels( train_notumor_dir , 3 )

In [124]:
x_validation_glioma , y_validation_glioma = load_images_and_labels( validation_glioma_dir , 0 )
x_validation_meningioma , y_validation_meningioma = load_images_and_labels( validation_meningioma_dir , 1 )
x_validation_pituitary , y_validation_pituitary = load_images_and_labels( validation_pituitary_dir , 2 )
x_validation_notumor , y_validation_notumor = load_images_and_labels( validation_notumor_dir , 3 )

In [125]:
x_train = np.array(x_train_glioma + x_train_meningioma + x_train_pituitary + x_train_notumor)
y_train = np.array(y_train_glioma + y_train_meningioma + y_train_pituitary + y_train_notumor)

x_validation = np.array(x_validation_glioma + x_validation_meningioma + x_validation_pituitary + x_validation_notumor)
y_validation = np.array(y_validation_glioma + y_validation_meningioma + y_validation_pituitary + y_validation_notumor)

In [126]:
def shuffle(images , labels):
    combined = list(zip(images , labels))
    np.random.shuffle(combined)
    shuffled_images , shuffled_labels = zip(*combined)
    return np.array(shuffled_images) , np.array(shuffled_labels)

In [127]:
x_train , y_train = shuffle(x_train , y_train)
x_validation , y_validation = shuffle(x_validation , y_validation)

In [128]:
x_train = x_train.astype('float32') / 255.0
x_validation = x_validation.astype('float32') / 255.0

In [129]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(128, 128 , 3)),


    
  

    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),


    tf.keras.layers.Dense(4, activation='softmax')
])

model.summary()

Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_13 (Flatten)            │ (None, 49152)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_48 (Dense)                │ (None, 256)            │    12,583,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_35 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_49 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_36 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_50 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_37 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_51 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,624,580 (48.16 MB)

 Trainable params: 12,624,580 (48.16 MB)

 Non-trainable params: 0 (0.00 B)

In [130]:
opt = tf.keras.optimizers.Adam(learning_rate=0.00001)

In [131]:
callback = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    min_delta=0,
    patience=5,
    verbose=0,
    mode="auto",
    baseline=None,
    restore_best_weights=False,
    start_from_epoch=0,
)

In [132]:
model.compile(optimizer=opt,
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

if len(x_train) > 0 and len(y_train) > 0 :
  history = model.fit(x_train , y_train ,
                      epochs=55,
                      batch_size=Batch_size,
                      callbacks=[callback],
                      validation_data=(x_validation, y_validation))


Epoch 1/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.3041 - loss: 1.3892 - val_accuracy: 0.4075 - val_loss: 1.3119
Epoch 2/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3455 - loss: 1.3345 - val_accuracy: 0.5175 - val_loss: 1.2770
Epoch 3/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3675 - loss: 1.3124 - val_accuracy: 0.5319 - val_loss: 1.2593
Epoch 4/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3966 - loss: 1.2876 - val_accuracy: 0.5806 - val_loss: 1.2254
Epoch 5/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4359 - loss: 1.2413 - val_accuracy: 0.6106 - val_loss: 1.1820
Epoch 6/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4529 - loss: 1.2241 - val_accuracy: 0.6181 - val_loss: 1.1746
Epoch 7/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4743 - loss: 1.2016 - val_accuracy: 0.6263 - val_loss: 1.1374
Epoch 8/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4859 - loss: 1.1770 - val_accuracy: 0

In [133]:
model.save("model_2.keras")